# 48. Near and far stations: waveforms, spectrograms and spectral roll-off

Pressure (infraBSU `DD0`) and vertical ground velocity (Trillium `DHZ`) at a near station (B01,
1.5 km) and a far station (B29, 5.6 km), with spectrograms, and the spectral centroid through the
launch as a measure of how the frequency content falls as the vehicle climbs. Arrival times are
also mapped to source time (`cfg.source_geometry`) so the roll-off can be read against altitude.

Outputs (`cfg.OUT_DIR`): `near_far.png`, `near_far_centroid.png`, `near_far_centroid.csv`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import spectrogram, welch
from obspy import Stream

import artemis2_config as cfg

L = cfg.LAUNCHTIME
OUT = cfg.OUT_DIR
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False})
STATIONS = ['B01', 'B29']
T0, T1 = -10, 140          # s after launch shown
stations = cfg.station_table()

In [ ]:
client = cfg.sds_client()
raw = Stream()
for sta in STATIONS:
    raw += client.get_waveforms('1R', sta, '00', 'DD0', cfg.STARTTIME, cfg.ENDTIME)
    raw += (cfg.read_restored('1R.B29.00.DHZ') if sta == 'B29'
            else client.get_waveforms('1R', sta, '00', 'DHZ', cfg.STARTTIME, cfg.ENDTIME))
raw.merge()
st = cfg.instrument_correct(raw)
for tr in st:
    tr.filter('highpass', freq=0.5 if tr.stats.channel == 'DD0' else 1.0, corners=4, zerophase=True)
print(st)

## Combined figure

In [ ]:
def get(sta, cha):
    return st.select(station=sta, channel=cha)[0]

fig = plt.figure(figsize=(13, 11))
gs = fig.add_gridspec(4, 2, height_ratios=[1, 1.4, 1, 1.4], hspace=0.35, wspace=0.18)
for col, (cha, unit, scale, clim) in enumerate([('DD0', 'Pressure (Pa)', 1, (-30, 30)),
                                                 ('DHZ', 'Velocity (mm/s)', 1e3, (-110, -70))]):
    ymax = max(np.abs(get(s, cha).slice(L + T0, L + T1).data).max() for s in STATIONS) * scale
    for k, sta in enumerate(STATIONS):
        tr = get(sta, cha)
        s = tr.slice(L + T0, L + T1)
        ax = fig.add_subplot(gs[2 * k, col])
        ax.plot(s.times() + T0, s.data * scale, color='#333333', lw=0.4)
        ax.set(xlim=(T0, T1), ylim=(-1.05 * ymax, 1.05 * ymax), ylabel=unit)
        ax.set_title(f"{sta} ({stations.distance_km[sta]:.1f} km), {'infraBSU' if cha == 'DD0' else 'Trillium'} "
                     f"{cha}; peak {np.abs(s.data).max() * scale:.3g}", fontsize=9, loc='left')
        ax.tick_params(labelbottom=False)
        ax2 = fig.add_subplot(gs[2 * k + 1, col], sharex=ax)
        fs = tr.stats.sampling_rate
        f, t, Sxx = spectrogram(s.data.astype(float), fs, nperseg=512, noverlap=384, scaling='density')
        db = 10 * np.log10(Sxx + 1e-30) + (0 if cha == 'DD0' else 0)
        im = ax2.pcolormesh(t + T0, f, db, shading='auto', cmap='magma',
                            vmin=np.percentile(db, 5), vmax=np.percentile(db, 99.5))
        ax2.set(yscale='log', ylim=(0.5 if cha == 'DD0' else 1, 225), ylabel='Frequency (Hz)')
        fig.colorbar(im, ax=ax2, pad=0.01, label='dB re 1 ' + ('Pa$^2$/Hz' if cha == 'DD0' else '(m/s)$^2$/Hz'))
        if k == 1:
            ax2.set_xlabel('Time after launch (s)')
        else:
            ax2.tick_params(labelbottom=False)
fig.savefig(OUT / 'near_far.png', dpi=200, bbox_inches='tight')

## Spectral centroid through the launch

In [ ]:
WIN = 4.0
rows = []
for tr in st:
    sta, cha = tr.stats.station, tr.stats.channel
    fs = tr.stats.sampling_rate
    geo = cfg.source_geometry(sta)
    noise = tr.slice(L - 600, L - 600 + WIN).data.astype(float).std()
    for t0 in np.arange(0, 200, 1.0):
        x = tr.slice(L + t0, L + t0 + WIN).data.astype(float)
        if x.std() < 10 * noise:
            continue
        f, p = welch(x, fs, nperseg=512)
        m = (f >= (0.5 if cha == 'DD0' else 1.0)) & (f <= 225)
        tm = t0 + WIN / 2
        j = min(np.searchsorted(geo.t_obs.to_numpy(), tm), len(geo) - 1)
        rows.append(dict(station=sta, channel=cha, t_arrival=tm,
                         centroid_hz=np.sum(f[m] * p[m]) / np.sum(p[m]),
                         t_source=geo.t_s.iloc[j] if tm >= geo.t_obs.iloc[0] else np.nan,
                         alt_km=geo.alt_km.iloc[j] if tm >= geo.t_obs.iloc[0] else np.nan))
cen = pd.DataFrame(rows)
cen.to_csv(OUT / 'near_far_centroid.csv', index=False)
COL = {'B01': '#2a78d6', 'B29': '#eb6834'}
fig, ax = plt.subplots(1, 2, figsize=(12, 4), sharey=False)
for a, (cha, lab) in zip(ax, (('DD0', 'Pressure'), ('DHZ', 'Vertical velocity'))):
    for sta, g in cen[cen.channel == cha].groupby('station'):
        a.plot(g.t_source, g.centroid_hz, color=COL[sta], lw=1, label=f"{sta} ({stations.distance_km[sta]:.1f} km)")
    a.set(xlabel='Source time after launch (s)', ylabel='Spectral centroid (Hz)', title=lab, yscale='log')
    a.legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(OUT / 'near_far_centroid.png', dpi=200)
for (sta, cha), g in cen.groupby(['station', 'channel']):
    a = g[g.t_source.between(10, 30)].centroid_hz.median(); b = g[g.t_source.between(60, 100)].centroid_hz.median()
    print(f"{sta} {cha}: median centroid {a:.1f} Hz at source times 10-30 s, {b:.1f} Hz at 60-100 s")